# Lab 25-03 · CVD Regimes, Dopant Diffusion, CMP and Gauge Noise

**Predict first**, then run. The handout is [README.md](README.md).

In [1]:
import math
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import numpy as np
from scipy import optimize
import units as u

## Part 1 · CVD regimes

*Predict* the temperature at which $k_s = h_g$ for Example 25-03.1's process.

In [2]:
tc = np.linspace(500, 1100, 300)
ks = np.array([u.arrhenius(0.05, 650, 1.5, t) for t in tc])
v = [u.cvd_rate_nm_min(k, 2.0, 3e16, 5e22) for k in ks]
print(f"ks = hg at {u.crossover_temperature(0.05, 650, 1.5, 2.0):.0f} C")
plt.figure()
plt.semilogy(1000 / (tc + 273.15), v)
plt.xlabel("1000/T (1/K)"); plt.ylabel("growth rate (nm/min)"); plt.grid(True, which="both")
plt.show()

ks = hg at 875 C


## Part 2 · Implant and anneal

*Predict* the junction depth after 30 min at 1000 °C, including the surface.

In [3]:
x = np.arange(0, 600.25, 0.25)
dose, rp, drp, nb = 1e15, 30.0, 12.0, 1e18
p0 = u.image_solution(x, dose, rp, drp, 0, 0)
fd = u.diffuse_fd(p0, 0.25, u.boron_d(1000), 1800)
img = u.image_solution(x, dose, rp, drp, u.boron_d(1000), 1800)
spike = u.image_solution(x, dose, rp, drp, u.boron_d(1050), 1.0)
xj = lambda prof: optimize.brentq(lambda z: np.interp(z, x, prof) - nb, rp, 590)
print(f"as implanted: {u.junction_depth(dose, rp, drp, nb):.1f} nm; spike: {xj(spike):.1f} nm")
print(f"furnace: FD {xj(fd):.2f} nm, image method {xj(img):.2f} nm; surface {img[0]:.3g} cm^-3")
plt.figure()
for prof, lab in ((p0, "as implanted"), (spike, "spike 1050 C, 1 s"), (fd, "furnace 1000 C, 30 min")):
    plt.semilogy(x, np.maximum(prof, 1e15), label=lab)
plt.axhline(nb, color="0.5", ls=":")
plt.xlim(0, 400); plt.ylim(1e16, 1e21)
plt.xlabel("depth (nm)"); plt.ylabel("boron (cm^-3)"); plt.legend(); plt.grid(True, which="both")
plt.show()

as implanted: 70.9 nm; spike: 72.2 nm
furnace: FD 243.74 nm, image method 243.74 nm; surface 9.8e+19 cm^-3


## Part 3 · CMP

*Predict* the time to remove a 300 nm step at 25% density.

In [4]:
rr = u.preston_rate_nm_min(2.4e-13, 2 * 6895, 1.0)
print(f"blanket {rr:.0f} nm/min; 300 nm step at 25%: {u.cmp_step_time_min(300, rr, 0.25):.3f} min")
dens = np.linspace(0.1, 1.0, 91)
plt.figure()
plt.plot(dens, [u.cmp_step_time_min(300, rr, d) for d in dens])
plt.xlabel("raised-feature area fraction"); plt.ylabel("time to remove 300 nm step (min)"); plt.grid(True)
plt.show()

blanket 199 nm/min; 300 nm step at 25%: 0.378 min


## Part 4 · Gauge noise

*Predict* the apparent $C_{pk}$ with a 0.05 nm gauge.

In [5]:
m = u.simulate_measurements(20.3, 0.40, 0.05)
cpk_sim = min(21.5 - m.mean(), m.mean() - 18.5) / (3 * m.std())
print(f"formula {u.apparent_cpk(20.3, 18.5, 21.5, 0.40, 0.05):.3f}; simulated {cpk_sim:.3f}")
sm = np.linspace(0, 0.4, 81)
plt.figure()
plt.plot(sm, [u.apparent_cpk(20.3, 18.5, 21.5, 0.40, s) for s in sm])
plt.xlabel("gauge sigma (nm)"); plt.ylabel("apparent Cpk"); plt.grid(True)
plt.show()

formula 0.992; simulated 0.994
